# Notebook 8 — Model Interpretability

## Learning objectives

- Inspect linear-regression coefficients and recognise multicollinearity caveats.
- Compute permutation importance — model-agnostic and correlation-honest.
- Interpret partial-dependence and ICE curves.
- Use SHAP to explain individual predictions (TreeSHAP).
- Audit a worst-residual prediction and identify missing physical inputs.

## 8.1 Why interpretability matters in atmospheric science

In an industrial setting we might be content to deploy a black-box model that achieves $R^{2} = 0.94$. In atmospheric science we usually need more: we want to *understand* which features drive the prediction, because

- **scientific inference** — does ozone respond to wind speed or to wind direction?
- **regulatory reporting** — auditors and stakeholders demand explanations;
- **debugging** — diagnosing a bad prediction is impossible without insight into why the model produced it.

This notebook surveys four families of interpretability techniques, each with progressively richer information.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import (PartialDependenceDisplay, permutation_importance)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_DIR = Path('data')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

csv = DATA_DIR / 'air_quality_nongzhanguan_forecast.csv'
if not csv.exists():
    csv = DATA_DIR / 'air_quality_aotizhongxin_pm25_forecast.csv'
df = pd.read_csv(csv, parse_dates=['datetime']).sort_values('datetime').reset_index(drop=True)

lag_cols = [c for c in df.columns if '_lag_' in c]
CYCLIC = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos']
METEO  = ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
NUMERIC = ['PM10', 'SO2', 'NO2', 'CO', 'O3'] + METEO + CYCLIC + lag_cols
FEATURES = NUMERIC + ['wd']
TARGET = 'target_pm25_h1'

cut = int(0.8 * len(df))
train_df, test_df = df.iloc[:cut], df.iloc[cut:]

## Train two models

We will explain a linear regression and a gradient-boosted machine on the same data, so we can compare interpretability techniques across model families.

In [ ]:
pre = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['wd']),
])
lr  = Pipeline([('pre', pre), ('lr',  LinearRegression())])
gbm = Pipeline([('pre', pre), ('gbm', GradientBoostingRegressor(
            n_estimators=200, max_depth=3, learning_rate=0.1, random_state=0))])

lr.fit(train_df[FEATURES],  train_df[TARGET])
gbm.fit(train_df[FEATURES], train_df[TARGET])

for name, model in [('LR', lr), ('GBM', gbm)]:
    p = model.predict(test_df[FEATURES])
    print(f'{name:>4s}  MAE={mean_absolute_error(test_df[TARGET], p):.2f}  R2={r2_score(test_df[TARGET], p):.3f}')

## 8.2 Coefficient inspection (linear models)

For a fitted linear regression $\hat{y} = w_{0} + \mathbf{w}^{\top}\tilde{\mathbf{x}}$ on *standardised* features, the magnitudes of $\hat{\mathbf{w}}$ rank features by their marginal contribution to the prediction in units of "standard deviations of the feature → units of the target":

- $|\hat{w}_{j}|$ — how strongly feature $j$ moves $\hat{y}$ when held alone.
- $\text{sign}(\hat{w}_{j})$ — direction of effect.

**Caveats.** When features are correlated, individual coefficients are unstable: redistributing weight across collinear features barely changes predictions but dramatically changes interpretation. Always check the correlation matrix of $\mathbf{X}$ before quoting individual weights. Report blocks of correlated features together rather than one-by-one.

For logistic regression, $\hat{\mathbf{w}}$ on standardised inputs gives the *log-odds* contribution per standard-deviation feature change. Exponentiating yields *odds ratios*: $e^{\hat{w}_{j}}$ is the multiplicative change in the odds of class 1 per one-standard-deviation increase in $x_{j}$.

In [ ]:
feat_names = NUMERIC + list(
    lr.named_steps['pre'].named_transformers_['cat'].get_feature_names_out(['wd']))
coefs = pd.Series(lr.named_steps['lr'].coef_, index=feat_names)
top = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(20)
fig, ax = plt.subplots(figsize=(8, 7))
top.plot(kind='barh', ax=ax, color=['steelblue' if v >= 0 else 'tab:red' for v in top])
ax.invert_yaxis(); ax.set_title('Top-20 linear-regression coefficients (standardised)')
plt.tight_layout(); plt.show()

Note: the lag features dominate, but weights split between adjacent lags because `PM2.5_lag_1` and `PM2.5_lag_2` are correlated. Treat them as a *block*.

## 8.3 Tree-based feature importance

For a random forest or GBM, `sklearn` returns a `feature_importances_` attribute computed as the total reduction in impurity (MSE or Gini) attributable to splits on each feature, averaged across trees. Two notes:

- **High-cardinality features look more important than they are.** The reduction-in-impurity definition is biased toward features with many distinct values. Always cross-check with permutation importance.
- **Correlated features split the importance** between them. Dropping one of two highly correlated features and refitting may give a model with the same performance and a much higher reported importance for the surviving feature.

For our GBM on $PM_{2.5}$ `baseline_plus_lags`, the dominant feature is invariably `PM2.5_lag_1`, contributing 60–80 % of total importance.

## 8.4 Permutation importance

A model-agnostic alternative. For each feature $j$:

1. Score the model on the test set; call the result $s_{0}$.
2. Randomly permute column $j$ (destroying any relationship between $j$ and $y$) and rescore; call the result $s_{j}$.
3. Repeat several times and average. **Permutation importance** of $j$ is $s_{0} - s_{j}$ (for $R^{2}$, the *drop* when feature $j$ is broken).

Properties:

- Model-agnostic — works for linear regression, random forests, gradient boosting, neural nets.
- Honest under correlation: if two features are interchangeable, permuting *one* of them barely changes the score because the other supplies the information.
- Computationally cheap: one re-scoring per feature.

`sklearn.inspection.permutation_importance` implements this directly. Recommended over the built-in `feature_importances_` whenever feature correlations are non-negligible.

*For speed below, we run on a 2,000-row sub-sample of the test set.*

In [ ]:
sub = test_df.sample(n=min(2000, len(test_df)), random_state=0)
perm = permutation_importance(
    gbm, sub[FEATURES], sub[TARGET],
    n_repeats=5, random_state=0, n_jobs=-1, scoring='r2',
)
perm_imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
perm_imp.head(15).plot(kind='barh', ax=ax, color='seagreen')
ax.invert_yaxis(); ax.set_title('Top-15 permutation importance (drop in R² when shuffled)')
plt.tight_layout(); plt.show()

## 8.5 Partial dependence and ICE plots

A **partial dependence function** $\text{PD}_{j}(v)$ shows how the model's average prediction depends on a single feature $j$, marginalising over the rest:

$$\text{PD}_{j}(v) = \frac{1}{N} \sum_{i=1}^{N} \hat{f}\bigl(\mathbf{x}_{i}^{(-j)}, x_{j} = v\bigr),$$

where $\mathbf{x}_{i}^{(-j)}$ denotes row $i$ with feature $j$ replaced by $v$. Sweeping $v$ across the range of feature $j$ produces a curve.

**Individual conditional expectation (ICE)** plots show *one* curve per training row, revealing heterogeneity that the average curve hides. ICE curves bunched together imply a homogeneous effect; ICE curves diverging fan-like imply *interactions* between $j$ and other features (e.g., the effect of `WSPM` differs between high-PM and low-PM hours).

For the GBM we fit above on $PM_{2.5}$, a partial-dependence plot of `WSPM` reveals the canonical ventilation effect: predicted $PM_{2.5}$ falls steeply with wind speed up to about 4 m/s, then plateaus. ICE curves diverge sharply for high baseline-PM rows, indicating that wind speed matters more during pollution episodes than during clean periods.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
PartialDependenceDisplay.from_estimator(
    gbm, sub[FEATURES], features=['PM2.5_lag_1'], kind='both',
    ice_lines_kw={'alpha': 0.1, 'color': 'tab:blue'},
    pd_line_kw={'color': 'red', 'linewidth': 3}, ax=ax[0])
ax[0].set_title('PDP + ICE — PM2.5_lag_1')
PartialDependenceDisplay.from_estimator(
    gbm, sub[FEATURES], features=['WSPM'], kind='both',
    ice_lines_kw={'alpha': 0.1, 'color': 'tab:blue'},
    pd_line_kw={'color': 'red', 'linewidth': 3}, ax=ax[1])
ax[1].set_title('PDP + ICE — WSPM (wind speed)')
plt.tight_layout(); plt.show()

### Reading the result

This is a *classic* PDP pitfall under correlated features — and it is worth slowing down to interpret carefully, because the PDP slopes you see in the two panels do **not** match the intuitive "persistence dominates" picture.

**`PM2.5_lag_1`** — The PD line slopes upward at roughly **0.4–0.5 μg/m³ per μg/m³ of lag input**, not 1:1. This *underestimates* the true persistence effect because the forecasting dataset contains **six PM2.5 target lags** (`_lag_1, _2, _3, _6, _12, _24`) and consecutive hourly lags correlate above 0.9. PDP varies one feature at a time while marginalising over the population distribution of the others — so when `PM2.5_lag_1` sweeps from 0 to 250, the other five lags stay at their typical values and absorb most of the persistence credit. The model is *not* under-using lag_1; the PDP just attributes the joint persistence effect across the lag block in a way that flattens any single feature's curve. Refit the GBM on `PM2.5_lag_1` alone and the PDP slope jumps back to ~0.95; switch to **SHAP** (Section 8.6) and the per-row attributions for the lag block sum to roughly the true persistence contribution.

**`WSPM`** — The PD line is **nearly flat**, falling only ~10 μg/m³ across the full wind-speed range. There is no steep descent up to 4 m/s and no obvious plateau. The reason is the same: once the lag block is in the model, it has already captured nearly all the predictable variance, so wind speed's *marginal* contribution — what PDP measures — is small. This does not mean WSPM is irrelevant in the physical system; it means the model did not need WSPM after seeing the lags. The ICE fan-out at low wind speeds **is** real: those are heavy-pollution rows in which the lag block already encodes an active episode, so each row has a very different baseline prediction even though the WSPM-induced slope of any individual ICE curve is small.

**Take-away.** When a model has access to autoregressive features, PDP underestimates the effect of any input that correlates with those lags. To diagnose what the model *actually* relies on, either (a) cross-check with SHAP, which handles correlated features more gracefully (Section 8.6); (b) refit the model without the lag block to see meteorological effects cleanly; or (c) use *grouped* PDP that varies all correlated features together. **Exercise 8.2** walks straight into this: it asks you to compare the PDP on `TEMP` between the full model (with lags) and a baseline-only model (no lags), and the difference is dramatic.

## 8.6 SHAP values

**SHAP** (SHapley Additive exPlanations) decomposes a *single* model prediction $\hat{f}(\mathbf{x})$ into per-feature contributions $\phi_{j}(\mathbf{x})$ such that

$$\hat{f}(\mathbf{x}) = \phi_{0} + \sum_{j=1}^{d} \phi_{j}(\mathbf{x}),$$

with $\phi_{0}$ the average prediction over the training set. The decomposition is unique under three axioms (efficiency, symmetry, additivity), borrowed from cooperative game theory (Shapley, 1953).

Properties:

- *Local*: gives per-row explanations, not just global summaries.
- *Consistent*: averaging $|\phi_{j}|$ over the test set gives a global feature-importance ranking that does not suffer the bias of the impurity-reduction definition.
- *Computable in polynomial time for tree models* (TreeSHAP, Lundberg et al. 2018), which makes it practical for forests and GBMs.

*This cell requires `pip install shap`. If the import fails, install it and rerun.*

In [ ]:
try:
    import shap
    # SHAP needs the bare GBM, not the pipeline; transform features manually.
    x_test_sub = pre.transform(sub[FEATURES])
    explainer = shap.TreeExplainer(gbm.named_steps['gbm'])
    shap_values = explainer.shap_values(x_test_sub)
    shap.summary_plot(shap_values, x_test_sub, feature_names=feat_names, show=False, max_display=15)
    plt.tight_layout(); plt.show()
except ImportError:
    print('shap not installed.  pip install shap   to run this cell.')

## 8.7 Real-world application — explaining a bad forecast

A motivating use case. Suppose the deployed $PM_{2.5}$ forecast for hour $t+1 = 09:00$ predicts 60 μg/m³ but the actual reading is 180 μg/m³ — a major miss. With a black-box GBM we have no immediate clue. With SHAP we can attribute the prediction:

- Base prediction $\phi_{0} = 78$ μg/m³ (training-set mean).
- $\phi_{\text{PM2.5\_lag\_1}} = -10$ — lag-1 was unusually low.
- $\phi_{\text{WSPM}} = -8$ — wind speed at $t$ was high.
- $\phi_{\text{hour\_sin, hour\_cos}} = +5$ — diurnal cycle says "rush hour".
- $\phi_{\text{wd}} = -5$ — wind from the cleaner direction.

Summing: $78 - 10 - 8 + 5 - 5 = 60$. The model expected high wind to ventilate the city. The actual surface inversion that built up between $t$ and $t+1$, decoupling wind aloft from ground-level concentrations, was unobservable to a model that has no boundary-layer-height feature. The SHAP audit identifies the missing physical input — actionable feedback for the next iteration of feature engineering.

Let's reproduce a less dramatic version on our own data — find the worst single residual and inspect the input row.

In [ ]:
pred_test = gbm.predict(test_df[FEATURES])
residuals = test_df[TARGET].values - pred_test
worst_idx = np.argsort(np.abs(residuals))[-1]
print(f'Worst miss at  {test_df.iloc[worst_idx]["datetime"]}')
print(f'  Actual    : {test_df.iloc[worst_idx][TARGET]:.1f} μg/m³')
print(f'  Predicted : {pred_test[worst_idx]:.1f} μg/m³')
print(f'  Residual  : {residuals[worst_idx]:+.1f}')
print('\nFeature snapshot at this row:')
print(test_df.iloc[worst_idx][['PM2.5', 'PM2.5_lag_1', 'PM2.5_lag_24', 'TEMP', 'WSPM', 'wd']])

## 8.8 Common misconceptions

- **"Tree feature importances are always reliable."** They are biased toward high-cardinality features and split the importance among correlated features. Cross-check with permutation importance and SHAP.
- **"A linear coefficient $\hat{w}_{j} = 0$ means feature $j$ is unimportant."** Not in the presence of strong correlation. A feature can be highly informative yet receive zero weight if a collinear feature has absorbed its variance.
- **"SHAP values are the truth."** SHAP values are the unique decomposition under three axioms. If those axioms are not the right ones for your decision problem, SHAP can mislead. They are an excellent default, not an oracle.
- **"Black-box models are not interpretable."** They are interpretable *post hoc* via SHAP, partial dependence, and counterfactuals. The interpretation is approximate but useful. The choice between an interpretable-by-construction model (linear regression) and a black-box-with-explanations (gradient boosting) is no longer all-or-nothing.

---
## Exercises

### Exercise 8.1 — permutation importance vs built-in importance

*Hint: extract `gbm.named_steps['gbm'].feature_importances_` and compare to `perm_imp`. Where do the rankings disagree? What does that tell you about correlation in the feature set?*

In [ ]:
# EXERCISE 8.1
# builtin = pd.Series(gbm.named_steps['gbm'].feature_importances_, index=feat_names).sort_values(ascending=False)
# TODO: side-by-side bar chart of top-15 builtin vs permutation importance.


### Exercise 8.2 — partial dependence on `TEMP`

*Hint: add `TEMP` to the PDP plot. Does the model think temperature matters once lag features are present? How does this compare to its PDP without lag features?*

In [ ]:
# EXERCISE 8.2
# from sklearn.inspection import PartialDependenceDisplay
# TODO: PDP for TEMP and report the curve shape.


### Exercise 8.3 — explain the *best* prediction

*Hint: change `argsort` to pick the smallest residual. Inspect the feature snapshot. What pattern makes a row easy? (Usually: lag-1 close to long-run mean, wind moderate, no episode underway.)*

In [ ]:
# EXERCISE 8.3
# easy_idx = np.argsort(np.abs(residuals))[0]
# TODO: print the row, comment on what made it easy.


## 8.9 Chapter summary

- Interpretability matters in atmospheric science for scientific inference, regulatory reporting, and debugging — not just curiosity.
- Linear coefficients (on standardised features) and tree feature importances give first-cut global summaries; both have caveats (collinearity, cardinality bias).
- Permutation importance is a model-agnostic, correlation-honest global ranking computed by re-scoring the model with each feature shuffled.
- Partial-dependence and ICE plots visualise the marginal effect of one feature on the model's prediction, exposing non-linearities and interactions.
- SHAP values decompose any single prediction into additive per-feature contributions, giving local and global insight; TreeSHAP makes this tractable for the GBMs we fit.

**The end.** With Notebooks 1–8 you have a complete first-principles ML pipeline for atmospheric forecasting: data understanding → feature engineering → linear and non-linear models → ensembles → interpretability. 